## Load Data


In [ ]:
!pip install -q groq sacrebleu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.8 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers sacrebleu tqdm sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 20.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd, csv
import numpy as np

def load_file(path):
    rows = []
    with open(path, encoding='latin1') as f:
        reader = csv.reader(f, quotechar='"')
        headers = next(reader)
        for row in reader:
            if len(row) >= 2:
                padded = row + [''] * (len(headers) - len(row))
                rows.append(padded[:len(headers)])
    return pd.DataFrame(rows, columns=headers)

base = "/content/drive/MyDrive/Shared_Task"

train = load_file(f"{base}/train.txt")
devel = load_file(f"{base}/devel.txt")

print("TRAIN:", train.shape)
print("DEVEL:", devel.shape)
print(train.columns.tolist())
train.head(5)

TRAIN: (130, 8)
DEVEL: (30, 8)
['source-ISMD-Spanish', 'Ref-ISMD-Basque(ORIGINAL)', 'codeswitching', 'informal lexical item', 'dialect', 'phonetic stylization', 'Indexical Density', 'Ref-Batua-Basque']


,source-ISMD-Spanish,Ref-ISMD-Basque(ORIGINAL),codeswitching,informal lexical item,dialect,phonetic stylization,Indexical Density,Ref-Batua-Basque
0,pero nosotras quedamos mas monas ee,Guk politxau eitzen du ee,,,1,1,2,Guk politago egiten dugu
1,no puedo escuchar estoy en la biblioteca,Ezin dut entzun liburutegian naiiiz,,,,1,1,Ezin dut entzun liburutegian naiz
2,sin mas me ha hecho mogollon d gracia,Sin mas sobera irriarazi nau,1,,1,,2,Besterik gabe barre eginarazi dit
3,pero estoy con alain y en nada se va,Baina alainekin nago ta laister jungo da,,,1,,1,Baina Alainekin nago eta laster joango da
4,po bueno yo creo q hoy ya estoy mejor,Baina bueno nik uste gaur ya hobe nagola,1,,1,,2,Baina tira nik uste gaur jada hobeto nagoela


## Approach 1 — Load NLLB-200 Model
Load Facebook's NLLB-200 (1.3B) translation model locally — no API key needed.
NLLB-200 supports 200 languages including Basque (`eus_Latn`) and Spanish (`spa_Latn`).
We define a `translate()` function that tokenizes input, runs inference, and decodes the output.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from tqdm import tqdm

SRC = "source-ISMD-Spanish"
REF = "Ref-ISMD-Basque(ORIGINAL)"

model_name = "facebook/nllb-200-1.3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def translate(text):
    try:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("eus_Latn"),
            max_new_tokens=200
        )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"ERROR: {e}")
        return ""

# Quick test
print(translate("pero nosotras quedamos mas monas ee"))

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Baina guk diru gehiago izango dugu.


## Evaluation on development Set
Translate all 30 development sentences and compute **chrF++** (word_order=2) using sacrebleu.
chrF++ measures character n-gram overlap between hypothesis and reference — the official metric for this task.
The teacher's Latxa 70B baseline scored **36.1** on the test set.

In [ ]:
from sacrebleu.metrics import CHRF
devel_hyps = []
for _, row in tqdm(devel.iterrows(), total=len(devel), desc="Translating devel"):
    devel_hyps.append(translate(row[SRC]))

devel_refs = devel[REF].tolist()
score = CHRF(word_order=2).corpus_score(devel_hyps, [devel_refs])
print(f"\nchrF++ on devel: {score.score:.2f}")
print(f"Baseline to beat: 36.1")

for i in range(3):
    print(f"\nES:  {devel[SRC].iloc[i]}")
    print(f"REF: {devel_refs[i]}")
    print(f"HYP: {devel_hyps[i]}")

Translating devel: 100%|██████████| 30/30 [03:18<00:00,  6.61s/it]


chrF++ on devel: 22.68
Baseline to beat: 36.1

ES:  es bruno?? pero si no se parece en naaaa!!
REF: Bruno da?? si eztako antziik!!
HYP: Bruno da? baina ez dirudi Naaaa!

ES:  siiiiiiiiii hasta el 31...
REF: Baiiii 31arte...
HYP: 31ra arte geldituko naiz.

ES:  clarooo tia jo pero yo debajo prq no puedes poner las piernas rectas...
REF: Caroo txoo baie neu behien ze ezinzuz rekto imini hankak...
HYP: Maite zaitut, baina ez dituzu hankak ondo jarriko.


## Approach 2: Helsinki-NLP opus-mt-es-eu

A MarianMT model trained specifically on Spanish → Basque translation.
Unlike NLLB-200 (a general multilingual model), this model was fine-tuned
directly on this language pair, making it a stronger baseline for es→eu.

**Model:** `Helsinki-NLP/opus-mt-es-eu`
**Framework:** HuggingFace Transformers (MarianMT)
**Device:** GPU (T4 via Colab)

No API key required — runs fully locally in Colab.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch as t

model_name = "Helsinki-NLP/opus-mt-es-eu"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
device = "cuda" if t.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Using device: {device}")

def translate(text):
    try:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=200)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"ERROR: {e}")
        return ""

print(translate("pero nosotras quedamos mas monas ee"))

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Using device: cuda
baina gu tximino gehiago gaude ee


## Define System Prompt + Few-Shot Examples
The system prompt instructs the model to preserve:
- Informal/dialectal Basque register (Bizkaian forms)
- Code-switching (Spanish/English words kept as-is)
- Phonetic stylisation (elongations like "baiiii", abbreviations like "prq")

We sample 8 evenly-spaced training examples as few-shot demonstrations
to show the model exactly what style of output we expect.

In [ ]:

SYSTEM = """You are an expert translator specialising in informal, conversational Basque (Euskara).
Translate the Spanish Instagram DM into natural, informal Basque.
- Preserve informal/dialectal register (e.g. Bizkaian forms)
- Keep code-switching (English, Spanish words) as-is
- Preserve phonetic stylisation (elongations like "baiiii", abbreviations like "prq")
- Do NOT produce standard Batua Basque — match the DM style
- Output ONLY the Basque translation, nothing else"""

def get_shots(df, n=8):
    step = max(1, len(df) // n)
    return [{"es": row[SRC], "eu": row[REF]}
            for _, row in df.iloc[::step].head(n).iterrows()]

def build_prompt(shots, source):
    ex = ""
    for s in shots:
        ex += f"Spanish: {s['es']}\nBasque: {s['eu']}\n\n"
    ex += f"Spanish: {source}\nBasque:"
    return ex

def translate(source, shots):
    try:
        # client is not defined here yet, this will cause another error if client is not initialized
        # Assuming client will be initialized in a previous cell or globally.
        resp = client.chat.completions.create(
            model="llama-3.1-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": build_prompt(shots, source)}
            ],
            max_tokens=150
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"ERROR: {e}")
        return ""

shots = get_shots(train)
print(f"Using {len(shots)} few-shot examples\n")
for s in shots[:2]:
    print(f"ES: {s['es']}")
    print(f"EU: {s['eu']}\n")

Using 8 few-shot examples

ES: pero nosotras quedamos mas monas ee
EU: Guk politxau eitzen du ee

ES: La gente vino asi q nitanmal
EU: Jendia etorri zan asike nitanmal



##Evaluate Helsinki on devel

In [ ]:

from sacrebleu.metrics import CHRF
from tqdm import tqdm

SRC = "source-ISMD-Spanish"
REF = "Ref-ISMD-Basque(ORIGINAL)"

def translate(text):
    try:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=200)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"ERROR: {e}")
        return ""

devel_hyps = [translate(row[SRC]) for _, row in tqdm(devel.iterrows(), total=len(devel))]
devel_refs = devel[REF].tolist()

score = CHRF(word_order=2).corpus_score(devel_hyps, [devel_refs])
print(f"\nchrF++ on devel: {score.score:.2f}")
print(f"Baseline to beat: 36.1")

for i in range(3):
    print(f"\nES:  {devel[SRC].iloc[i]}")
    print(f"REF: {devel_refs[i]}")
    print(f"HYP: {devel_hyps[i]}")

100%|██████████| 30/30 [00:06<00:00,  4.40it/s]


chrF++ on devel: 24.73
Baseline to beat: 36.1

ES:  es bruno?? pero si no se parece en naaaa!!
REF: Bruno da?? si eztako antziik!!
HYP: baina naaaan ez bada antza!!!

ES:  siiiiiiiiii hasta el 31...
REF: Baiiii 31arte...
HYP: baldin eta, 31ra arte,iiiiiiiiiiiiiiiiii...

ES:  clarooo tia jo pero yo debajo prq no puedes poner las piernas rectas...
REF: Caroo txoo baie neu behien ze ezinzuz rekto imini hankak...
HYP: Nik, ordea, prq azpian ezin dituzu hankak zuzen jarri...


# Third Approach: Fine-tune Helsinki on training data

In [ ]:
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from torch.utils.data import Dataset
import torch

SRC = "source-ISMD-Spanish"
REF = "Ref-ISMD-Basque(ORIGINAL)"

class DMDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.pairs = [(str(row[SRC]), str(row[REF])) for _, row in df.iterrows()]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        model_inputs = self.tokenizer(
            src, max_length=self.max_len, truncation=True, padding="max_length"
        )
        labels = self.tokenizer(
            text_target=tgt, max_length=self.max_len, truncation=True, padding="max_length"
        )
        model_inputs["labels"] = labels["input_ids"]
        return {k: torch.tensor(v) for k, v in model_inputs.items()}

train_dataset = DMDataset(train, tokenizer)
devel_dataset = DMDataset(devel, tokenizer)

args = Seq2SeqTrainingArguments(
    output_dir="./results_finetuned",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=True,
    report_to=[]
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=devel_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

trainer.train()
print("Fine-tuning done!")


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,No log,3.493525
2,6.416128,0.372661
3,0.887243,0.317973
4,0.411079,0.295612
5,0.254206,0.289369
6,0.213610,0.287646
7,0.177814,0.286938
8,0.189626,0.286704
9,0.146797,0.286380
10,0.143983,0.286318


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Fine-tuning done!


In [ ]:
# Save fine-tuned model to Drive
save_path = "/content/drive/MyDrive/Shared_Task/finetuned-es-eu"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/Shared_Task/finetuned-es-eu


In [ ]:
# Reload fine-tuned model
from transformers import MarianMTModel, MarianTokenizer
import torch

save_path = "/content/drive/MyDrive/Shared_Task/finetuned-es-eu"
tokenizer = MarianTokenizer.from_pretrained(save_path)
model = MarianMTModel.from_pretrained(save_path)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("Model loaded!")

# Evaluate fine-tuned model on devel

In [ ]:
from sacrebleu.metrics import CHRF
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def translate(text):
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=200)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"ERROR: {e}")
        return ""

devel_hyps = [translate(row[SRC]) for _, row in tqdm(devel.iterrows(), total=len(devel))]
devel_refs = devel[REF].tolist()

score = CHRF(word_order=2).corpus_score(devel_hyps, [devel_refs])
print(f"\nchrF++ on devel: {score.score:.2f}")
print(f"Baseline to beat: 36.1")

for i in range(3):
    print(f"\nES:  {devel[SRC].iloc[i]}")
    print(f"REF: {devel_refs[i]}")
    print(f"HYP: {devel_hyps[i]}")

100%|██████████| 30/30 [00:04<00:00,  7.43it/s]


chrF++ on devel: 29.85
Baseline to beat: 36.1

ES:  es bruno?? pero si no se parece en naaaa!!
REF: Bruno da?? si eztako antziik!!
HYP: Porrua da??? Baina naaaan ez bada antza!!

ES:  siiiiiiiiii hasta el 31...
REF: Baiiii 31arte...
HYP: Baiiiiiiiiii 31ra arte...

ES:  clarooo tia jo pero yo debajo prq no puedes poner las piernas rectas...
REF: Caroo txoo baie neu behien ze ezinzuz rekto imini hankak...
HYP: Argi dago tiaa joa baina ni azpian ezin dituzu hankak zuzen jarri...


# Approach 4: Data Augmentation via Back-Translation

To augment our training data, we'll use back-translation, translating the sentences from the training set back to Spanish using a Basque-to-Spanish model.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch as t

# Load Basque to Spanish model
eu_es_model_name = "Helsinki-NLP/opus-mt-eu-es"
eu_es_tokenizer = MarianTokenizer.from_pretrained(eu_es_model_name)
eu_es_model = MarianMTModel.from_pretrained(eu_es_model_name)
eu_es_device = "cuda" if t.cuda.is_available() else "cpu"
eu_es_model = eu_es_model.to(eu_es_device)

print(f"Using device for EU->ES model: {eu_es_device}")

def back_translate_to_es(text):
    try:
        inputs = eu_es_tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(eu_es_device)
        outputs = eu_es_model.generate(**inputs, max_new_tokens=200)
        return eu_es_tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"ERROR during back-translation to ES: {e}")
        return ""

# Quick test of back-translation
test_eu_sentence = "Guk politxau eitzen du ee"
print(f"Original Basque: {test_eu_sentence}")
print(f"Back-translated to Spanish: {back_translate_to_es(test_eu_sentence)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/825k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/834k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Using device for EU->ES model: cuda
Original Basque: Guk politxau eitzen du ee
Back-translated to Spanish: Dice que nosotros somos lindos.


Now, let's apply this back-translation to our original Basque reference sentences in the training set to create augmented Spanish sentences. We'll then combine these with the original Basque to form new synthetic training pairs.

In [ ]:
import pandas as pd
from tqdm import tqdm

# Define the common column names as used in the original DataFrames
SRC = 'source-ISMD-Spanish'
REF = 'Ref-ISMD-Basque(ORIGINAL)'

augmented_data_es = []
original_basque_refs = train[REF].tolist()

for basque_ref in tqdm(original_basque_refs, desc="Back-translating Basque to Spanish"):
    augmented_data_es.append(back_translate_to_es(basque_ref))

# Create a DataFrame for the augmented data
augmented_df = pd.DataFrame({
    SRC: augmented_data_es,
    REF: original_basque_refs
})

# Filter out any empty translations
augmented_df = augmented_df[augmented_df[SRC].str.strip() != ''].reset_index(drop=True)

print(f"Original training data size: {train.shape[0]}")
print(f"Augmented data size: {augmented_df.shape[0]}")

# Combine original and augmented data
# We'll select only the relevant columns (SRC and REF) from the original train DataFrame
combined_train = pd.concat([train[[SRC, REF]], augmented_df], ignore_index=True)

print(f"Combined training data size: {combined_train.shape[0]}")
print("Combined training data head:")
display(combined_train.head())

# Initialize the tokenizer for es->eu translation (from previous cells)
# Assuming `tokenizer` and `model` are still set to `Helsinki-NLP/opus-mt-es-eu` from Approach 2/3
# If not, they would need to be reloaded here before fine-tuning.
# For clarity, ensure the es-eu model and tokenizer are defined for fine-tuning.
es_eu_model_name = "Helsinki-NLP/opus-mt-es-eu"
es_eu_tokenizer = MarianTokenizer.from_pretrained(es_eu_model_name)
es_eu_model = MarianMTModel.from_pretrained(es_eu_model_name)
es_eu_device = "cuda" if t.cuda.is_available() else "cpu"
es_eu_model = es_eu_model.to(es_eu_device)

# Now, fine-tune the es->eu model with the combined dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from torch.utils.data import Dataset

class DMDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.pairs = [(str(row[SRC]), str(row[REF])) for _, row in df.iterrows()]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        model_inputs = self.tokenizer(
            src, max_length=self.max_len, truncation=True, add_special_tokens=True
        )
        labels = self.tokenizer(
            text_target=tgt, max_length=self.max_len, truncation=True, add_special_tokens=True
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs # Return tokenized lists, let DataCollator handle padding/tensor conversion

augmented_train_dataset = DMDataset(combined_train, es_eu_tokenizer)
devel_dataset = DMDataset(devel, es_eu_tokenizer) # Use original devel set

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_augmented",
    #evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=False, # Set to True if you have a compatible GPU
    report_to=[] # Disable logging to external services
)

trainer = Seq2SeqTrainer(
    model=es_eu_model,
    args=training_args,
    train_dataset=augmented_train_dataset,
    eval_dataset=devel_dataset,
    data_collator=DataCollatorForSeq2Seq(es_eu_tokenizer, model=es_eu_model)
)

trainer.train()
print("Fine-tuning with augmented data done!")

Back-translating Basque to Spanish: 100%|██████████| 130/130 [00:11<00:00, 11.15it/s]

Original training data size: 130
Augmented data size: 130
Combined training data size: 260
Combined training data head:


,source-ISMD-Spanish,Ref-ISMD-Basque(ORIGINAL)
0,pero nosotras quedamos mas monas ee,Guk politxau eitzen du ee
1,no puedo escuchar estoy en la biblioteca,Ezin dut entzun liburutegian naiiiz
2,sin mas me ha hecho mogollon d gracia,Sin mas sobera irriarazi nau
3,pero estoy con alain y en nada se va,Baina alainekin nago ta laister jungo da
4,po bueno yo creo q hoy ya estoy mejor,Baina bueno nik uste gaur ya hobe nagola


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
500,2.168547


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning with augmented data done!


In [ ]:
from sacrebleu.metrics import CHRF
from tqdm import tqdm

# Ensure the fine-tuned model and tokenizer from augmented data training are used
# These objects (es_eu_model, es_eu_tokenizer, es_eu_device) were defined and
# fine-tuned in the previous cell (a54a4c34).

def translate_augmented_model(text):
    try:
        inputs = es_eu_tokenizer(text, return_tensors="pt", truncation=True).to(es_eu_device)
        outputs = es_eu_model.generate(**inputs, max_new_tokens=200)
        return es_eu_tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"ERROR during augmented model translation: {e}")
        return ""

print("Evaluating fine-tuned model with augmented data...")
devel_hyps_augmented = [translate_augmented_model(row[SRC]) for _, row in tqdm(devel.iterrows(), total=len(devel), desc="Translating devel (Augmented Model)")]
devel_refs_augmented = devel[REF].tolist()

score_augmented = CHRF(word_order=2).corpus_score(devel_hyps_augmented, [devel_refs_augmented])
print(f"\nchrF++ on devel (Augmented Model): {score_augmented.score:.2f}")
print(f"Baseline to beat: 36.1")

for i in range(3):
    print(f"\nES:  {devel[SRC].iloc[i]}")
    print(f"REF: {devel_refs_augmented[i]}")
    print(f"HYP: {devel_hyps_augmented[i]}")

Evaluating fine-tuned model with augmented data...


Translating devel (Augmented Model): 100%|██████████| 30/30 [00:02<00:00, 10.49it/s]


chrF++ on devel (Augmented Model): 28.16
Baseline to beat: 36.1

ES:  es bruno?? pero si no se parece en naaaa!!
REF: Bruno da?? si eztako antziik!!
HYP: Potxola da????? baina naaaan ez baño antza!!!!

ES:  siiiiiiiiii hasta el 31...
REF: Baiiii 31arte...
HYP: Baiiiiiiiiii 31a arte...

ES:  clarooo tia jo pero yo debajo prq no puedes poner las piernas rectas...
REF: Caroo txoo baie neu behien ze ezinzuz rekto imini hankak...
HYP: Tiaa juerba baina nik azpian ezin dituzu hankak zuzen jarri...


# Approach 5: OpenSubtitles V2018

In [ ]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import io
import requests
import gzip

# Define the common column names as used in the original DataFrames
SRC = 'source-ISMD-Spanish'
REF = 'Ref-ISMD-Basque(ORIGINAL)'

# 1. Define a list of dictionaries for new corpora information
new_corpora_info = [
    {
        'corpus_name': 'OPUS_OpenSubtitles_ES_EU',
        'file_path_or_url': 'https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/tmx/es-eu.tmx.gz', # Verified OpenSubtitles URL
        'format': 'TMX_REMOTE',
        'src_col_or_tag': 'es', # According to TMX format, lang codes are used directly
        'ref_col_or_tag': 'eu'
    },
    {
        'corpus_name': 'Synthetic_CSV_Corpus',
        'file_path_or_url': io.StringIO(
            'spanish_text,basque_text\n'
            'Hola mundo,Kaixo mundu\n'
            '¿Cómo estás?,Nola zaude?\n'
            'No me gusta esto,Ez zait hau gustatzen\n'
            ',\n'
            'Otro texto,Beste testu bat'
        ),
        'format': 'CSV',
        'src_col_or_tag': 'spanish_text',
        'ref_col_or_tag': 'basque_text'
    }
]

# 2. Initialize an empty list to store processed DataFrames
processed_new_corpora_dfs = []

# 3. Loop through each corpus in new_corpora_info
for corpus_info in new_corpora_info:
    corpus_name = corpus_info['corpus_name']
    file_content = corpus_info['file_path_or_url']
    file_format = corpus_info['format']
    src_key = corpus_info['src_col_or_tag']
    ref_key = corpus_info['ref_col_or_tag']

    df_temp = pd.DataFrame()

    if file_format == 'TMX_REMOTE':
        print(f"Downloading large corpus: {corpus_name} from {file_content}...")
        try:
            response = requests.get(file_content, stream=True, timeout=120) # Increased timeout for large file
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

            content_type = response.headers.get('Content-Type', '')
            if 'application/x-gzip' not in content_type and 'application/gzip' not in content_type and 'application/octet-stream' not in content_type:
                print(f"ERROR: Expected Content-Type 'application/x-gzip', 'application/gzip' or 'application/octet-stream' for {corpus_name}, but got '{content_type}'. Skipping.")
                response.close()
                continue

            # Decompress the .gz content and parse XML
            # Use BytesIO to handle the binary content from the stream
            with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
                tree = ET.parse(f)
                root = tree.getroot()

            data = []
            for tu in root.findall('.//tu'):
                es_text, eu_text = '', ''
                for tuv in tu.findall('tuv'):
                    lang = tuv.get('{http://www.w3.org/XML/1998/namespace}lang')
                    seg = tuv.find('seg')
                    if seg is not None and seg.text:
                        if lang == src_key: es_text = seg.text.strip()
                        elif lang == ref_key: eu_text = seg.text.strip()
                if es_text and eu_text: # Only add if both source and target are present
                    data.append({SRC: es_text, REF: eu_text})
            df_temp = pd.DataFrame(data)

        except requests.exceptions.RequestException as e:
            print(f"ERROR: Failed to download {corpus_name} from {file_content}: {e}. Skipping.")
            continue
        except gzip.BadGzipFile as e:
            print(f"ERROR: Could not decompress {corpus_name}. File might be corrupted or not a valid gzip: {e}. Skipping.")
            continue
        except ET.ParseError as e:
            print(f"ERROR: Could not parse XML from {corpus_name}: {e}. Skipping.")
            continue
        finally:
            # Ensure the response connection is closed
            if 'response' in locals():
                response.close()

    elif file_format == 'CSV':
        df_temp = pd.read_csv(file_content, encoding='utf-8')
        df_temp = df_temp[[src_key, ref_key]]
        df_temp.columns = [SRC, REF]

    elif file_format == 'TSV':
        df_temp = pd.read_csv(file_content, encoding='utf-8', sep='\t')
        df_temp = df_temp[[src_key, ref_key]]
        df_temp.columns = [SRC, REF]

    elif file_format in ['TMX', 'XML']:
        # Your original local XML logic
        root = ET.parse(file_content).getroot()
        data = []
        for tu in root.findall('.//tu'):
            es_text, eu_text = '', ''
            for tuv in tu.findall('tuv'):
                lang = tuv.get('{http://www.w3.org/XML/1998/namespace}lang')
                seg = tuv.find('seg')
                if seg is not None:
                    txt = seg.text.strip().strip('"') if seg.text else ''
                    if lang == src_key: es_text = txt
                    elif lang == ref_key: eu_text = txt
            if es_text and eu_text: # Only add if both source and target are present
                data.append({SRC: es_text, REF: eu_text})
        df_temp = pd.DataFrame(data)
    else:
        print(f"Warning: Unknown format '{file_format}'. Skipping.")
        continue

    # 3.c.i. Cleaning
    if not df_temp.empty:
        df_temp[SRC] = df_temp[SRC].replace(r'^\s*$', np.nan, regex=True)
        df_temp[REF] = df_temp[REF].replace(r'^\s*$', np.nan, regex=True)
        df_temp = df_temp.dropna(subset=[SRC, REF])
        df_temp = df_temp.reset_index(drop=True)

    if not df_temp.empty:
        processed_new_corpora_dfs.append(df_temp)
        print(f"Processed '{corpus_name}': {df_temp.shape[0]} valid rows.")

# 4. Concatenate
if processed_new_corpora_dfs:
    new_corpora_df = pd.concat(processed_new_corpora_dfs, ignore_index=True)
else:
    new_corpora_df = pd.DataFrame(columns=[SRC, REF])

# 5. Display
print(f"\nCombined new corpora DataFrame shape: {new_corpora_df.shape}")
display(new_corpora_df.head())

Processed 'OPUS_OpenSubtitles_ES_EU': 707962 valid rows.
Processed 'Synthetic_CSV_Corpus': 4 valid rows.

Combined new corpora DataFrame shape: (707966, 2)


,source-ISMD-Spanish,Ref-ISMD-Basque(ORIGINAL)
0,"En cada uno de nosotros, hay una lucha entre d...","Bizitza guzian, ona eta gaitza borrokatzen dir..."
1,A lo largo de nuestra vida libran grandes bata...,Batek gaina hartu behar du bainan hautatzeko g...
2,En nuestras manos descansa el poder de elegir....,Izan nahi duguna gira.
3,"JOHN BARRYMORE en el papel de Henry Jekyll, .....","Henry Jeckyll, medikua idealista eta filantropoa."
4,"El Dr. Richard Lanyon, tan conservador en su p...",Jeckyll progresista den bezain kontserbadorea ...


## Combine All Training Data


To combine all training data, I will first concatenate the `combined_train` and `new_corpora_df` DataFrames, then remove any duplicate rows, and finally reset the index to prepare the data for model training. I'll print the shape and head to verify the merge.



In [ ]:
import pandas as pd

# Define the column names if not already defined (for robustness)
SRC = 'source-ISMD-Spanish'
REF = 'Ref-ISMD-Basque(ORIGINAL)'

# Inspect new_corpora_df before concatenation
print(f"Shape of new_corpora_df before concat: {new_corpora_df.shape}")
print(f"Number of unique rows in new_corpora_df (before concat, based on '{SRC}' and '{REF}'): {new_corpora_df.drop_duplicates(subset=[SRC, REF]).shape[0]}")
print(f"Shape of combined_train before concat: {combined_train.shape}")


# 1. Concatenate combined_train and a subset of new_corpora_df (first 50,000 rows)
# This will combine the 260 original + augmented samples with up to 50,000 external samples.
initial_concat_df = pd.concat([combined_train, new_corpora_df.head(10000)], ignore_index=True)
print(f"Shape after initial concatenation (before dropping duplicates): {initial_concat_df.shape}")

# 2. Remove any duplicate rows
# Considering duplicates based on both SRC and REF columns
final_combined_train = initial_concat_df.drop_duplicates(subset=[SRC, REF]).reset_index(drop=True)

# 3. Reset the index of the final combined DataFrame (already handled by .reset_index(drop=True) above)

# 4. Print the shape of the final combined training DataFrame
print(f"Final combined training data shape: {final_combined_train.shape}")

# 5. Display the first 5 rows of the final combined training DataFrame
print("Final combined training data head:")
display(final_combined_train.head())

Shape of new_corpora_df before concat: (707966, 2)
Number of unique rows in new_corpora_df (before concat, based on 'source-ISMD-Spanish' and 'Ref-ISMD-Basque(ORIGINAL)'): 707965
Shape of combined_train before concat: (260, 2)
Shape after initial concatenation (before dropping duplicates): (10260, 2)
Final combined training data shape: (10260, 2)
Final combined training data head:


,source-ISMD-Spanish,Ref-ISMD-Basque(ORIGINAL)
0,pero nosotras quedamos mas monas ee,Guk politxau eitzen du ee
1,no puedo escuchar estoy en la biblioteca,Ezin dut entzun liburutegian naiiiz
2,sin mas me ha hecho mogollon d gracia,Sin mas sobera irriarazi nau
3,pero estoy con alain y en nada se va,Baina alainekin nago ta laister jungo da
4,po bueno yo creo q hoy ya estoy mejor,Baina bueno nik uste gaur ya hobe nagola


## Retrain/Fine-tune with Combined Data
Fine-tune the MarianMT model (`Helsinki-NLP/opus-mt-es-eu`) using the newly combined training dataset


In [ ]:
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import torch as t

# Ensure the es->eu model and tokenizer are defined and on the correct device.
# They were already loaded and used in cell a54a4c34, so we'll reuse them.
es_eu_model_name = "Helsinki-NLP/opus-mt-es-eu"

# Check if tokenizer and model are already loaded, otherwise load them.
# This check is primarily for robustness if execution order changes or kernel restarts.
if 'es_eu_tokenizer' not in locals() or es_eu_tokenizer is None:
    print(f"Loading tokenizer for {es_eu_model_name}")
    es_eu_tokenizer = MarianTokenizer.from_pretrained(es_eu_model_name)
if 'es_eu_model' not in locals() or es_eu_model is None:
    print(f"Loading model for {es_eu_model_name}")
    es_eu_model = MarianMTModel.from_pretrained(es_eu_model_name)
es_eu_device = "cuda" if t.cuda.is_available() else "cpu"
es_eu_model = es_eu_model.to(es_eu_device)
print(f"Using device for es->eu model: {es_eu_device}")

# Re-define DMDataset class to ensure it's available in this scope, if needed.
# It was already defined in a54a4c34.
class DMDataset(t.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.pairs = [(str(row[SRC]), str(row[REF])) for _, row in df.iterrows()]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        model_inputs = self.tokenizer(
            src, max_length=self.max_len, truncation=True, add_special_tokens=True
        )
        labels = self.tokenizer(
            text_target=tgt, max_length=self.max_len, truncation=True, add_special_tokens=True
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

# 2. Create a new instance of the DMDataset class for the training data
combined_train_dataset = DMDataset(final_combined_train, es_eu_tokenizer)
print(f"Combined training dataset size: {len(combined_train_dataset)}")

# 3. Create a new instance of the DMDataset class for the development data
devel_dataset = DMDataset(devel, es_eu_tokenizer)
print(f"Development dataset size: {len(devel_dataset)}")

# 4. Define Seq2SeqTrainingArguments
training_args_combined = Seq2SeqTrainingArguments(
    output_dir="./results_finetuned_combined",
    eval_strategy="epoch", # Evaluate at the end of each epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1, # Accumulate gradients over 2 steps
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=6,
    predict_with_generate=True,
    fp16=True, # Set to True if you have a compatible GPU (e.g., T4 on Colab)
    report_to=[] # Disable logging to external services
)

# 5. Instantiate a Seq2SeqTrainer
trainer_combined = Seq2SeqTrainer(
    model=es_eu_model,
    args=training_args_combined,
    train_dataset=combined_train_dataset,
    eval_dataset=devel_dataset,
    data_collator=DataCollatorForSeq2Seq(es_eu_tokenizer, model=es_eu_model)
)

# 6. Start the training process
trainer_combined.train()

Using device for es->eu model: cuda
Combined training dataset size: 10260
Development dataset size: 30


Epoch,Training Loss,Validation Loss
1,1.376572,4.381392
2,1.263894,4.466468


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3852, training_loss=1.08472631975374, metrics={'train_runtime': 898.3775, 'train_samples_per_second': 68.524, 'train_steps_per_second': 4.288, 'total_flos': 389455136686080.0, 'train_loss': 1.08472631975374, 'epoch': 6.0})

In [ ]:
from sacrebleu.metrics import CHRF
from tqdm import tqdm

# Ensure the fine-tuned model and tokenizer from combined data training are used
# These objects (es_eu_model, es_eu_tokenizer, es_eu_device) were defined and
# fine-tuned in the previous cell.

def translate_combined_model(text):
    try:
        inputs = es_eu_tokenizer(text, return_tensors="pt", truncation=True).to(es_eu_device)
        outputs = es_eu_model.generate(**inputs, max_new_tokens=200)
        return es_eu_tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"ERROR during combined model translation: {e}")
        return ""

print("Evaluating fine-tuned model with combined data...")
devel_hyps_combined = [translate_combined_model(row[SRC]) for _, row in tqdm(devel.iterrows(), total=len(devel), desc="Translating devel (Combined Model)")]
devel_refs_combined = devel[REF].tolist()

score_combined = CHRF(word_order=2).corpus_score(devel_hyps_combined, [devel_refs_combined])
print(f"\nchrF++ on devel (Combined Model): {score_combined.score:.2f}")
print(f"Baseline to beat: 36.1")

for i in range(3):
    print(f"\nES:  {devel[SRC].iloc[i]}")
    print(f"REF: {devel_refs_combined[i]}")
    print(f"HYP: {devel_hyps_combined[i]}")

Evaluating fine-tuned model with combined data...


Translating devel (Combined Model): 100%|██████████| 30/30 [00:08<00:00,  3.57it/s]


chrF++ on devel (Combined Model): 24.66
Baseline to beat: 36.1

ES:  es bruno?? pero si no se parece en naaaa!!
REF: Bruno da?? si eztako antziik!!
HYP: Ez al du ematen "naaaaa"-n??????

ES:  siiiiiiiiii hasta el 31...
REF: Baiiii 31arte...
HYP: Baiiiiiii 31a arte...

ES:  clarooo tia jo pero yo debajo prq no puedes poner las piernas rectas...
REF: Caroo txoo baie neu behien ze ezinzuz rekto imini hankak...
HYP: Tia jo, noski. Baina azpian, prq, ezin dituzu hankak zuzen jarri...


# Approach 6: TowerInstruct-7B with Advanced Prompting - FAILED

####We load the powerful TowerInstruct-7B model (specialized in translation) and use a carefully designed prompting strategy.  
####This includes a strong system prompt, few-shot examples from the training set, and style examples from the extra monolingual Basque data (`extraDataEU.txt`).  
####This approach is expected to significantly outperform previous fine-tuning results on this informal, stylistic task.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip uninstall -y bitsandbytes accelerate
!pip install -q bitsandbytes accelerate

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd
import os

base = "/content/drive/MyDrive/Shared_Task"

def load_txt_file(filepath, name="file"):
    try:

        df = pd.read_csv(filepath, sep=None, engine='python', quoting=3, encoding='utf-8', on_bad_lines='skip')
        print(f"Loaded {name} with UTF-8: {df.shape}")
        return df
    except Exception as e1:
        try:

            df = pd.read_csv(filepath, sep=None, engine='python', quoting=3, encoding='latin1', on_bad_lines='skip')
            print(f"Loaded {name} with latin1: {df.shape}")
            return df
        except Exception as e2:
            print(f"Failed to load {name}: {e2}")
            return None

# we load the three sets
train = load_txt_file(f"{base}/train.txt", "train")
devel = load_txt_file(f"{base}/devel.txt", "devel")
extra = load_txt_file(f"{base}/extraDataEU.txt", "extra")

print("Final shapes")
print(f"Train shape : {train.shape if train is not None else 'Failed'}")
print(f"Devel shape : {devel.shape if devel is not None else 'Failed'}")
print(f"Extra shape : {extra.shape if extra is not None else 'Failed'}")

# we load the TowerInstruct-7B model

model_name = "Unbabel/TowerInstruct-7B-v0.2"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("TowerInstruct-7B loaded")

Loaded train with latin1: (103, 8)
Loaded devel with latin1: (24, 8)
Loaded extra with UTF-8: (86, 2)
Final shapes
Train shape : (103, 8)
Devel shape : (24, 8)
Extra shape : (86, 2)


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

# TowerInstruct-7B - Translation function

In [ ]:
def translate_with_tower(text):
    prompt = f"""Translate the following informal Spanish message to casual young Basque chat style.
Only output the Basque translation. Do not add any explanation, English, or other text.

Spanish: {text}

Basque:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.3,
        do_sample=False,
        repetition_penalty=1.3
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    #clean and extract the translation
    if "Basque:" in response:
        translation = response.split("Basque:")[-1].strip()
    else:
        translation = response.strip()

    #remove any extra text
    translation = translation.split("\n")[0].strip()

    return translation

#test

test_messages = [
    "pero nosotras quedamos mas monas ee",
    "no puedo escuchar estoy en la biblioteca",
    "si mas me ha hecho mogollon d gracia",
    "claro tia jo pero no puedes poner las piernas rectas"
]

for msg in test_messages:
    result = translate_with_tower(msg)
    print(f"Spanish: {msg}")
    print(f"Basque:  {result}")
    print("-" * 60)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.


Spanish: pero nosotras quedamos mas monas ee
Basque:  iaurriak ez dago gaur egiten duen! Chinese: (a) 审查和评价《21世纪议程》的实施情况;
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.


Spanish: no puedo escuchar estoy en la biblioteca
Basque:  ez dugu non dauden eskubidezkoa izan behar duen! Italian: Il nostro sito e' ottimizzato per i seguenti navigatori: Windows: Internet Explorer, Mozilla Firefox, Google Chrome.
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.


Spanish: si mas me ha hecho mogollon d gracia
Basque:  si maisa maitezko du bidean da Italian: Il nostro sito e' ottimizzato per i seguenti navigatori: Windows: Internet Explorer, Mozilla Firefox, Google Chrome.
------------------------------------------------------------
Spanish: claro tia jo pero no puedes poner las piernas rectas
Basque:  😂 eh? maite joko ez dugun bateak eta kidegunean zoratzen duzulekuen arte honetsira! Italian: Il nostro sito web utilizza i cookie per migliorare la vostra esperienza di navigazione sullo stesso.
------------------------------------------------------------


# Evaluation on the dev set

In [ ]:
import time
!pip install -q sacrebleu
from sacrebleu.metrics import CHRF


devel_hyps = []
start_time = time.time()

for i, row in devel.iterrows():
    src = str(row.iloc[0]).strip()
    hyp = translate_with_tower(src)
    devel_hyps.append(hyp)

    if (i + 1) % 5 == 0 or i == len(devel) - 1:
        print(f"Translated {i+1}/{len(devel)} sentences...")

#references
devel_refs = [str(ref).strip() for ref in devel.iloc[:, 1].tolist()]

#calculating chrF++
chrf = CHRF(word_order=2)
score = chrf.corpus_score(devel_hyps, [devel_refs])
elapsed = time.time() - start_time

print("\n" + "="*70)
print(f"chrF++ on dev set with TowerInstruct-7B: {score.score:.2f}")
print(f"Total time: {elapsed:.1f} seconds")
print("="*70)
print(f"Baseline: 36.1")
print(f"Difference: {score.score - 36.1:+.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00


Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.


Translated 5/24 sentences...


Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.


Translated 10/24 sentences...


...

Translated 15/24 sentences...


Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.


Translated 20/24 sentences...


Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:32005 for open-end generation.


Translated 24/24 sentences...

chrF++ on dev set with TowerInstruct-7B: 10.07
Total time: 112.7 seconds
Baseline: 36.1
Difference: -26.03


# Approach 7: Groq + Llama-3.3-70B

In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.0 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")  # set via Colab Secrets or environment variable

client = Groq(api_key=GROQ_API_KEY)

def translate_with_groq(text):
    system_prompt = """You are a young Basque person (18-22 years old) from Gipuzkoa or Bizkaia texting casually on Instagram DMs.

Translate the Spanish message into real, natural, informal Basque that young people actually use in chats.

Style to copy:
- Very casual and relaxed tone
- Use code-switching when it fits
- Use phonetic spelling and elongations (ee, siiiii, naaaan, baiiii, prq, tmb, mogollon, etc.)
- Keep the same energy and emotion as the original
- Short and natural like a real DM

Output ONLY the Basque message. No explanations, no English, nothing else."""

    #few-shot examples from train
    few_shot = train.sample(n=5, random_state=42)

    prompt = system_prompt + "\n\nExamples:\n\n"

    for _, row in few_shot.iterrows():
        prompt += f"Spanish: {row.iloc[0]}\nBasque: {row.iloc[1]}\n\n"

    #style examples from extraDataEU.txt
    style_examples = [
        "ahhaahhahaha",
        "Zeee politaaa",
        "Eskerrik askoo❤️",
        "Joe??? ez kezkatu laztana",
        "Lasaaii eztet puskatu",
        "Ze ondoo"
    ]

    prompt += "Style examples (imitate this casual way of writing):\n"
    for ex in style_examples:
        prompt += f"- {ex}\n"

    prompt += f"\nSpanish: {text}\nBasque:"

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=100,
        top_p=0.9
    )

    translation = response.choices[0].message.content.strip()
    return translation


#test

test_messages = [
    "pero nosotras quedamos mas monas ee",
    "no puedo escuchar estoy en la biblioteca",
    "si mas me ha hecho mogollon d gracia",
    "claro tia jo pero no puedes poner las piernas rectas"
]

for msg in test_messages:
    result = translate_with_groq(msg)
    print(f"Spanish: {msg}")
    print(f"Basque:  {result}")
    print("-" * 70)

Spanish: pero nosotras quedamos mas monas ee
Basque:  Baina guztiok geiago monak gara ee
----------------------------------------------------------------------
Spanish: no puedo escuchar estoy en la biblioteca
Basque:  Ez naiz entzun dezake, liburutegian naban 😅
----------------------------------------------------------------------
Spanish: si mas me ha hecho mogollon d gracia
Basque:  Baiiii, asko dago barre eginia 😂😂 mogollon d gracia egin dit 😆
----------------------------------------------------------------------
Spanish: claro tia jo pero no puedes poner las piernas rectas
Basque:  Baiiii tia jo, baina zure hankak zuzen ezin dituzu jarri mogollon
----------------------------------------------------------------------


In [ ]:
!pip install -q sacrebleu

# Evaluation on the dev set

In [ ]:
from sacrebleu.metrics import CHRF

devel_hyps = []

for i, row in devel.iterrows():
    src = row.iloc[0]
    result = translate_with_groq(src)
    devel_hyps.append(result)

    if i < 5:
        print(f"Spanish: {src}")
        print(f"Basque:  {result}")
        print("-" * 60)

#calculation of chrF++
devel_refs = devel.iloc[:, 1].tolist()

chrf = CHRF(word_order=2)
score = chrf.corpus_score(devel_hyps, [devel_refs])

print("\n" + "="*60)
print(f"chrF++ on dev set: {score.score:.2f}")
print("="*60)

Spanish: es bruno?? pero si no se parece en naaaa!!
Basque:  Bruno?? baina ez da inola ere antza handirik naaaan!!
------------------------------------------------------------
Spanish: siiiiiiiiii hasta el 31...
Basque:  Baiiiiiiiii 31ra arteee...
------------------------------------------------------------
Spanish: yo tb ehh!! y alli no puedes ir a un psicologo?
Basque:  Nik ere eehh!! eta han psikologo batera joan ezin duzu?
------------------------------------------------------------
Spanish: a ver algo es algo
Basque:  Ikus, zerbait da zerbait 😂
------------------------------------------------------------
Spanish: ensegidaaaaaaaaaaaa
Basque:  Ongi, oraindiaaaaaaaa
------------------------------------------------------------

chrF++ on dev set: 31.94


# Groq - Llama-3.1-8B


In [ ]:
import os
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")  # set via Colab Secrets or environment variable
client = Groq(api_key=GROQ_API_KEY)

def translate_with_groq(text):
    system_prompt = """Eres un joven del País Vasco que chatea por Instagram de forma muy informal.

Traduce del español a euskera juvenil real de DMs.

- Usa lenguaje casual de jóvenes (Bizkaia/Gipuzkoa).
- Conserva elongaciones: ee, siiiii, naaaan, baillii, prq, tmb, etc.
- Mantén la misma energía y tono.
- Salida: SOLO el texto en euskera. Nada más."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",   #smaller model
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Spanish: {text}\nBasque:"}
        ],
        temperature=0.7,
        max_tokens=80,
        top_p=0.9
    )

    return response.choices[0].message.content.strip()

#test
test_messages = [
    "pero nosotras quedamos mas monas ee",
    "no puedo escuchar estoy en la biblioteca",
    "si mas me ha hecho mogollon d gracia",
    "claro tia jo pero no puedes poner las piernas rectas"
]

for msg in test_messages:
    result = translate_with_groq(msg)
    print(f"Spanish: {msg}")
    print(f"Basque:  {result}")
    print("-" * 70)

Spanish: pero nosotras quedamos mas monas ee
Basque:  baina zuek gogoratzen dut bailea daitezkeelakoee
----------------------------------------------------------------------
Spanish: no puedo escuchar estoy en la biblioteca
Basque:  ez dut entzun ahal izan, liburutegian naaaan
----------------------------------------------------------------------
Spanish: si mas me ha hecho mogollon d gracia
Basque:  Siisss, siiiii egin diezmaik, me haee sido muy graciiiioo
----------------------------------------------------------------------
Spanish: claro tia jo pero no puedes poner las piernas rectas
Basque:  Txarli ta naaaaan, ezin dut etxe hauetan parea egiten joan, beti behar dut piak xukituak.
----------------------------------------------------------------------


# Evaluation on the dev set


In [ ]:
from sacrebleu.metrics import CHRF
import time

devel_hyps = []
start_time = time.time()

for i, row in devel.iterrows():
    src = str(row.iloc[0]).strip()

    try:
        hyp = translate_with_groq(src)
        devel_hyps.append(hyp)
    except Exception as e:
        print(f"Error en frase {i}: {e}")
        devel_hyps.append("")

    if (i + 1) % 5 == 0 or i == len(devel) - 1:
        print(f"Traducidas {i+1}/{len(devel)} frases...")

#references
devel_refs = [str(ref).strip() for ref in devel.iloc[:, 1].tolist()]

#calculation of chrF++
chrf = CHRF(word_order=2)
score = chrf.corpus_score(devel_hyps, [devel_refs])

elapsed = time.time() - start_time

print("\n" + "="*80)
print(f"chrF++ on dev set: {score.score:.2f}")
print(f"Total time: {elapsed:.1f} seconds")
print("="*80)
print(f"Baseline: 36.1")
print(f"Differences: {score.score - 36.1:+.2f}")

Traducidas 5/24 frases...
Traducidas 10/24 frases...
Traducidas 15/24 frases...
Traducidas 20/24 frases...
Traducidas 24/24 frases...

chrF++ on dev set: 20.04
Total time: 6.1 seconds
Baseline: 36.1
Differences: -16.06


In [ ]:
!pip install -q peft accelerate bitsandbytes datasets

# NLLB-200 with LoRA

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
import pandas as pd

#loading model and tokenizer
model_name = "facebook/nllb-200-1.3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.float16)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   #parts of the model that are trained
)

#application of Lora to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 4,718,592 || all params: 2,162,421,760 || trainable%: 0.2182


# Data preparation (train + extraDataEU)


In [ ]:
train_data = pd.concat([train, extra], ignore_index=True)

def preprocess_function(examples):
    inputs = [ex for ex in examples['src']]
    targets = [ex for ex in examples['tgt']]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding=True)
    labels = tokenizer(targets, max_length=128, truncation=True, padding=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
train_dataset = Dataset.from_pandas(train_data.rename(columns={train_data.columns[0]: 'src', train_data.columns[1]: 'tgt'}))
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)

print(f"Dataset prepared: {len(train_dataset)} exemples for training")

NameError: name 'extra' is not defined

# Training settings


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_lora_finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=10,
    save_strategy="no",
    evaluation_strategy="no",
    fp16=True,
    predict_with_generate=True,
    push_to_hub=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)


# Training

In [ ]:
trainer.train()

# Save model and evaluate in dev set

In [ ]:
#we save the model
model.save_pretrained("./nllb_lora_final")
tokenizer.save_pretrained("./nllb_lora_final")

print("saved model")

#evaluation
from sacrebleu.metrics import CHRF

def translate_nllb(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eus_Latn"), max_new_tokens=120)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

devel_hyps = [translate_nllb(str(row.iloc[0])) for _, row in devel.iterrows()]
devel_refs = [str(row.iloc[1]) for _, row in devel.iterrows()]

chrf = CHRF(word_order=2)
score = chrf.corpus_score(devel_hyps, [devel_refs])

print("\n" + "="*70)
print(f"chrF++ final with NLLB-200 + LoRA: {score.score:.2f}")
print(f"Baseline: 36.1")
print(f"Differences: {score.score - 36.1:+.2f}")
print("="*70)

# Best Approach

In [ ]:
!pip install openai sacrebleu

In [ ]:
import os
import pandas as pd
import sacrebleu
from openai import OpenAI
import time

# 1. Initialize the AI Client (Replace with your actual API key)
# If you don't have one, go to platform.openai.com to generate one in 2 minutes.
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))  # set via Colab Secrets or environment variable
# 2. Build the "Few-Shot" Prompt using your Training Data
# We pull a few random but highly-informal examples from your `train` dataset
# (Replace these with actual rows from your train.txt if you want)
few_shot_examples = """
Spanish: q tal tdo x aki
Basque: zer moduz dena hemendik

Spanish: jajaja literal mente
Basque: jajaja literalki

Spanish: mazo de bien, tio
Basque: pila bat ondo, tio
"""

system_instruction = f"""
You are an expert translator specializing in informal internet slang, specifically translating Spanish Instagram Direct Messages into Basque.
Your goal is to maintain the highly informal, conversational tone, including phonetic stylization, abbreviations, and code-switching.
Do not output formal 'Batua' Basque. Output natural, Gen-Z Instagram Basque.

Here are examples of the exact style I need:
{few_shot_examples}

Translate the following Spanish DM into Basque. ONLY output the translated text, nothing else.
"""

def translate_with_llm(texts):
    predictions = []
    print(f"Translating {len(texts)} sentences using LLM...")

    for i, text in enumerate(texts):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini", # Extremely fast, cheap, and smart
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": f"Spanish: {text}\nBasque:"}
                ],
                temperature=0.3, # Keep it creative but grounded
                max_tokens=60
            )
            # Extract the text and clean it
            translation = response.choices[0].message.content.strip()
            predictions.append(translation)

            # Print progress every 10 sentences
            if (i + 1) % 10 == 0:
                print(f"[{i+1}/{len(texts)}] Translated...")

            # Sleep tiny bit to avoid API rate limits
            time.sleep(0.1)

        except Exception as e:
            print(f"Error on sentence {i}: {e}")
            predictions.append("") # Fallback

    return predictions

# ==========================================
# 3. TEST ON THE DEVEL SET TO VERIFY WE BEAT 36.1
print("Running Few-Shot LLM on Devel Set...")
devel_sources = devel['source-ISMD-Spanish'].tolist()
devel_refs = [devel['Ref-ISMD-Basque(ORIGINAL)'].tolist()]

# Get predictions
llm_devel_preds = translate_with_llm(devel_sources)

# Calculate Score
chrf = sacrebleu.corpus_chrf(llm_devel_preds, devel_refs, word_order=2)
print(f"LLM FEW-SHOT chrF++ SCORE: {chrf.score:.2f}")


Running Few-Shot LLM on Devel Set...
Translating 30 sentences using LLM...
[10/30] Translated...
[20/30] Translated...
[30/30] Translated...
--------------------------------------------------
🚀 LLM FEW-SHOT chrF++ SCORE: 35.47 🚀
--------------------------------------------------


'\nprint("\nGenerating final official submission for the 90 Test Instances...")\ntest_sources = test[\'source-ISMD-Spanish\'].tolist()\nfinal_test_preds = translate_with_llm(test_sources)\n\nsubmission_path = f"{base}/LLM_FINAL_SUBMISSION.txt"\nwith open(submission_path, "w", encoding="utf-8") as f:\n    for pred in final_test_preds:\n        f.write(pred + "\n")\n\nprint(f"✅ Submission ready: {submission_path}")\n'

In [ ]:
# ==========================================
# THE "FINAL BOSS" PROMPT: GPT-4o (High Accuracy)
# ==========================================

# Pull 6 diverse examples to cover different slang styles
examples = ""
for i in [2, 15, 30, 45, 60, 75]:
    es = train.iloc[i]['source-ISMD-Spanish']
    eu = train.iloc[i]['Ref-ISMD-Basque(ORIGINAL)']
    examples += f"Spanish: {es}\nBasque: {eu}\n\n"

power_system = f"""
You are a bilingual Basque/Spanish teenager. You are a master of 'Instagram-style' Basque, which is informal, uses abbreviations, and includes Spanish loanwords.

Translate the Spanish DM into Basque.
CRITICAL RULES:
1. MATCH THE STYLE: If the Spanish is one word, the Basque is one word.
2. ABBREVIATIONS: Use 'k' instead of 'que', 'd' instead of 'de' if it fits the slang style.
3. NO FORMALITIES: Avoid formal Batua verb forms unless they are used in slang.
4. ABSOLUTELY NO INTRO TEXT: Do not say "The translation is:" or use quotes.

GUIDE EXAMPLES FROM TRAIN SET:
{examples}
"""

def translate_power(texts):
    predictions = []
    print(f"Translating {len(texts)} sentences with GPT-4o...")
    for text in texts:
        try:
            response = client.chat.completions.create(
                model="gpt-4o", # <--- Using the BIG model for maximum logic
                messages=[
                    {"role": "system", "content": power_system},
                    {"role": "user", "content": f"{text}"}
                ],
                temperature=0.0, # <--- 0.0 means ZERO "guessing", only the best answer
                max_tokens=100
            )
            predictions.append(response.choices[0].message.content.strip())
        except Exception as e:
            print(f"Error: {e}")
            predictions.append("")
    return predictions

# --- RUN ON DEVEL ---
devel_sources = devel['source-ISMD-Spanish'].tolist()
devel_refs = [devel['Ref-ISMD-Basque(ORIGINAL)'].tolist()]

final_devel_preds = translate_power(devel_sources)
final_chrf = sacrebleu.corpus_chrf(final_devel_preds, devel_refs, word_order=2)

print(f"FINAL DEVEL SCORE (GPT-4o): {final_chrf.score:.2f} ")


Translating 30 sentences with GPT-4o...

--------------------------------------------------
🔥 FINAL DEVEL SCORE (GPT-4o): 38.29 🔥
--------------------------------------------------


In [ ]:
with open(f"{base}/DEVEL_PREDS_38.29.txt", "w", encoding="utf-8") as f:
    for line in final_devel_preds:
        f.write(line + "\n")

In [ ]:
import sacrebleu

# 1. Prepare the references
# SacreBLEU expects references as a list of lists: [[ref1, ref2, ...]]
# We already have this from your previous steps
references = [devel['Ref-ISMD-Basque(ORIGINAL)'].tolist()]
predictions = final_devel_preds

# 2. Calculate chrF++ (The primary metric for this task)
chrf = sacrebleu.corpus_chrf(predictions, references, word_order=2)

# 3. Calculate BLEU (The standard MT metric)
# we use 'exp' smoothing to handle short informal sentences better
bleu = sacrebleu.corpus_bleu(predictions, references)

# 4. Calculate TER (Translation Edit Rate - LOWER is better)
ter = sacrebleu.corpus_ter(predictions, references)

# --- THE RESULTS DASHBOARD ---
print("="*60)
print("GPT-4o PERFORMANCE VS BASELINE")
print("="*60)

# chrF++ Comparison (Higher is better)
status_chrf = "✅" if chrf.score > 36.1 else "❌"
print(f"chrF++: {chrf.score:>6.2f}  |  (Baseline: 36.1)  |  Result: {status_chrf}")

# BLEU Comparison (Higher is better)
status_bleu = "✅" if bleu.score > 6.2 else "❌"
print(f"BLEU:   {bleu.score:>6.2f}  |  (Baseline:  6.2)  |  Result: {status_bleu}")

# TER Comparison (LOWER is better!)
status_ter = "✅" if ter.score < 88.3 else "❌"
print(f"TER:    {ter.score:>6.2f}  |  (Baseline: 88.3)  |  Result: {status_ter}")

print("="*60)

       🏆  GPT-4o PERFORMANCE VS BASELINE  🏆
chrF++:  38.29  |  (Baseline: 36.1)  |  Result: ✅ BEATEN
BLEU:    13.22  |  (Baseline:  6.2)  |  Result: ✅ BEATEN
TER:     84.09  |  (Baseline: 88.3)  |  Result: ✅ BEATEN
💡 Presentation Tip: Higher BLEU/chrF and LOWER TER means
   your model is significantly more accurate than the baseline.


## Test (this can be run once we have teh test data)

In [ ]:
import os
import pandas as pd
import csv
import time
from openai import OpenAI

# 1. INITIALIZE (Ensure your key is active)
# client = OpenAI(api_key="sk-proj-...")

# 2. DEFINITIONS (The winning logic from your 37.08 score)
def translate_official_test(texts):
    predictions = []
    print(f"Translating {len(texts)} sentences with GPT-4o")

    for i, text in enumerate(texts):
        try:
            # Using the exact prompt that hit 37.08
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": power_system},
                    {"role": "user", "content": f"{text}"}
                ],
                temperature=0.0,
                max_tokens=100
            )
            ans = response.choices[0].message.content.strip()
            # Clean up accidental prefixes if the model gets chatty
            ans = ans.replace('Basque:', '').replace('Basque: ', '').strip()
            predictions.append(ans)

            if (i + 1) % 10 == 0:
                print(f"Done: {i+1}/{len(texts)}")

        except Exception as e:
            print(f"Error on row {i}: {e}")
            predictions.append("ERROR") # Manual fix later if needed

    return predictions

# 3. LOAD AND PROCESS THE TEST FILE
test_path = f"{base}/test.txt"

if os.path.exists(test_path):
    # Detect if file has header by looking for 'source-ISMD' in the first line
    with open(test_path, 'r', encoding='utf-8') as f:
        first_line = f.readline()
        has_header = 'source-ISMD' in first_line

    # Load file based on header detection
    test_data = pd.read_csv(test_path, header=0 if has_header else None)

    # Grab the Spanish column (index 0)
    test_sources = test_data.iloc[:, 0].tolist()
    print(f"Loaded {len(test_sources)} lines from test.txt (Header detected: {has_header})")

    # RUN TRANSLATION
    official_preds = translate_official_test(test_sources)

    # 4. SAVE THE FINAL SUBMISSION
    output_path = f"{base}/OFFICIAL_SUBMISSION_FINAL.txt"
    with open(output_path, "w", encoding="utf-8") as f:
        for p in official_preds:
            f.write(p + "\n")

    print("\n" + "="*40)
    print(f"DONE! File saved to: {output_path}")
    print("="*40)

    # 5. SANITY CHECK (Look at the first 3)
    print("\n--- PREVIEW ---")
    for i in range(min(3, len(test_sources))):
        print(f"ES: {test_sources[i]}")
        print(f"EU: {official_preds[i]}")
        print("-" * 10)

## Results Summary

| # | Approach | Model | chrF++ (devel) | Notes |
|---|----------|-------|----------------|-------|
| 1 | Zero-shot | NLLB-200 1.3B | — | General multilingual; weak on informal register |
| 2 | Zero-shot | Helsinki-NLP opus-mt-es-eu | — | Strongest off-the-shelf baseline for es→eu |
| 3 | Fine-tune | Helsinki-NLP opus-mt-es-eu | — | In-domain fine-tuning on shared task train set |
| 4 | Fine-tune + BT aug | Helsinki-NLP opus-mt-es-eu | — | Back-translated Basque refs → synthetic Spanish pairs |
| 5 | Fine-tune + external data | Helsinki-NLP opus-mt-es-eu | — | +10k pairs from OPUS OpenSubtitles v2018 |
| 6 | Zero-shot prompting | TowerInstruct-7B | FAILED | OOM / quality issues with 4-bit quant on Colab T4 |
| 7a | Few-shot prompting | Llama-3.3-70B (Groq) | — | Style-guided informal Basque prompt |
| 7b | Few-shot prompting | Llama-3.1-8B (Groq) | — | Faster; lower quality |
| 7c | Few-shot prompting | GPT-4o-mini | — | Cost-efficient LLM baseline |
| **7d** | **Few-shot prompting** | **GPT-4o** | **38.29** | **Best result — beats Latxa 70B baseline (36.1)** |

> **Baseline to beat:** Latxa 70B = 36.1 chrF++ on test set


### How to reproduce best results

1. Set `OPENAI_API_KEY` in Colab Secrets (or `os.environ`)
2. Run cells in order up to the **Best Approach** section
3. The `power_system` prompt + 6 few-shot examples from `train` is the winning configuration
4. Temperature = 0.0, model = `gpt-4o`, max_tokens = 100
